In [38]:
!pip install langchain-groq

In [64]:
import os
from getpass import getpass
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

In [65]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [154]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=1024
)

prompt = ChatPromptTemplate.from_messages([
    ("system","You are helpful AI assistant. You need to only provide answer from given context"),
    ("human","context:{context}, \n question: {message}"),
    ])

parser = StrOutputParser()

qa_chain = prompt | llm | parser

In [155]:
qa_chain.invoke({
    "context": "Capital of Pakistan is Lahore",
    "message": "What is the capital of Pakistan?"
})

'Lahore.'

In [156]:
prompt1 = ChatPromptTemplate.from_messages([
    ("system","You are helpful AI assistant."),
    ("human","History:{history}, query: {message}")
])

chain = prompt1 | llm | parser

In [157]:
history = ""
def chat(message):
  global history
  output = chain.invoke({
    "message": message,
    "history": history
  })
  history += "Human: "+message+"\n"
  history += "AI: "+output+"\n"
  return output

In [158]:
chat("My name is anas")

''

In [159]:
chat("what is my name?")

''

In [160]:
chat("what is last human message?")

''

In [161]:
print(history)

Human: My name is anas
AI: 
Human: what is my name?
AI: 
Human: what is last human message?
AI: 



In [162]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder

In [164]:
history_prompt = ChatPromptTemplate.from_messages([
    ("system","You are helpful AI assistant"),
    MessagesPlaceholder(variable_name="history"),
    ("human","{user_input}"),
])

store={}
def get_session_history(session_id: str):
  if session_id not in store:
     store[session_id] = InMemoryChatMessageHistory()
  return store[session_id]


chat_with_history = RunnableWithMessageHistory(
    history_prompt | llm | parser,
    get_session_history,
    input_messages_key="user_input",
    history_messages_key="history"
)


In [165]:
store

{}

In [166]:
config = {"configurable":{"session_id":"session_1"}}

In [167]:
chat_with_history.invoke({"user_input":"My name is Ali!"}, config = config)

'Nice to meet you,'

In [168]:
chat_with_history.invoke({"user_input":"What is my name!"}, config = config)

'Your name is Ali.'

In [169]:
store.pop("session_1")

InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Ali!', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you,', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name!', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Ali.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [170]:
def hist(session_id):
  return session_id
def runable_pass_through(fun, config: dict):
    history = fun(**config)

runable_pass_through(hist,config = {"session_id":"123123"})


In [172]:
import uuid

session_id = str(uuid.uuid4())

while True:
  message = input("message : ")
  if message == "exit":
    break
  else:
    output = chat_with_history.invoke({"user_input":message}, config = {"configurable":{"session_id":session_id}})
    print(output)


message : hi my name is anas
Hi Anas! Nice to meet you
message : what is my name ?
Your name is Anas.
message : exit
